# 텍스트 임베딩

건강 팁이 수백 개 쌓이면 "이 질문과 비슷한 내용"을 찾아야 합니다. 단어가 겹치는지로는 부족하고 의미로 비교해야 합니다. 임베딩(embedding)은 문장을 숫자 벡터로 바꿔 의미 사이의 거리를 계산할 수 있게 해 줍니다. 다음 노트북의 RAG에서 쓸 재료를 여기서 만듭니다.

**🎯 미션**

1. 건강 테마 문구를 `client.embeddings.create()`로 임베딩 벡터로 변환합니다.
2. 코사인 유사도(cosine similarity)로 어떤 문구가 서로 가까운지 확인합니다.

> **사전 준비**: [02장](../02-foundry-project/README.md)에서 **text-embedding-3-small** 모델이 배포되어 있어야 합니다.

> **안내문**: 이 노트북은 교육 목적의 예제입니다. 의학적 조언이 필요할 경우 반드시 전문가와 상담하세요.


## 1. 설정 및 환경 구성

In [ ]:
import os
from pathlib import Path

from dotenv import load_dotenv
from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient

# 상위 폴더의 .env 로드
load_dotenv(Path().absolute().parent / ".env")

endpoint = os.environ["PROJECT_ENDPOINT"]
chat_model = os.environ["MODEL_NAME"]                    # 배포 이름 (예: gpt-5-mini)
embedding_model = os.environ["TEXT_EMBEDDING_MODEL"]     # 배포 이름 (예: text-embedding-3-small)

# Entra ID(keyless) 인증 — 사전에 `az login` 필요
project = AIProjectClient(endpoint=endpoint, credential=DefaultAzureCredential())

# 프로젝트 범위 클라이언트 (chat/responses/agents 용)
client = project.get_openai_client()

# 임베딩 전용 클라이언트 — 프로젝트 범위 엔드포인트에는 /embeddings 라우트가 없어 리소스 범위를 쓴다
account_endpoint = endpoint.split("/api/projects/")[0]
embedding_client = project.get_openai_client(base_url=f"{account_endpoint}/openai/v1")
print("✅ AIProjectClient / OpenAI client 준비 완료")


## 2. 텍스트 임베딩

아래 건강 테마 문구들을 임베딩합니다. 출력은 각 문장을 의미론적 공간(semantic space)에서 표현하는 **숫자 벡터**입니다.

In [ ]:
text_phrases = [
    "An apple a day keeps the doctor away 🍎",
    "Quick 15-minute HIIT workout routine 🏋️",
    "Mindful breathing exercises 🧘",
]

response = embedding_client.embeddings.create(
    model=embedding_model,  # 임베딩 모델 "배포 이름"
    input=text_phrases,
)

for item in response.data:
    length = len(item.embedding)
    print(
        f"data[{item.index}]: length={length}, "
        f"[{item.embedding[0]:.6f}, {item.embedding[1]:.6f}, "
        f"..., {item.embedding[-2]:.6f}, {item.embedding[-1]:.6f}]"
    )
print(response.usage)


## 3. 유사도 확인해보기

임베딩의 쓸모는 **의미적 유사도**에 있습니다. 코사인 유사도로 어떤 문구가 서로 가까운지 확인해봅시다.

In [ ]:
import math

def cosine(a, b):
    dot = sum(x * y for x, y in zip(a, b))
    return dot / (math.sqrt(sum(x * x for x in a)) * math.sqrt(sum(y * y for y in b)))

vecs = [item.embedding for item in response.data]
query = "Best short high-intensity exercise for busy people"
q_vec = embedding_client.embeddings.create(model=embedding_model, input=[query]).data[0].embedding

for phrase, v in zip(text_phrases, vecs):
    print(f"{cosine(q_vec, v):.4f}  {phrase}")
print("\n→ HIIT 문구가 가장 높은 유사도를 보이면 성공입니다!")


## ✅ 미션 완료

**무엇을 만들었나:**

- ✓ 건강 테마 문구를 표현하는 임베딩 벡터
- ✓ 코사인 유사도로 문구 사이 의미 거리를 비교하는 코드

임베딩 벡터를 검색 인덱스에 저장하면 **RAG**의 재료가 됩니다 → [03-basic-rag.ipynb](03-basic-rag.ipynb)

> 시작 전에 [README](README.md)의 **Azure AI Search 구성**을 먼저 완료하세요.
